In [ ]:
import os
import sys

root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root_path not in sys.path:
    sys.path.insert(0, root_path) 

from src.models import EmployeesOrm, ResumesOrm, Workload
from src.schemas import ResumesDTO, ResumesRelDTO, EmployeesDTO, EmployeesRelDTO
from src.database import sync_session, sync_engine

from sqlalchemy import select, func, and_, or_, Integer, cast
from sqlalchemy.orm import selectinload

In [8]:
with sync_session() as session:
    query = (
        select(EmployeesOrm)
        .limit(2)
    )

    res = session.execute(query)
    result_orm = res.scalars().all()
    print(f'\n\n{result_orm=}\n\n')
    result_dto = [EmployeesDTO.model_validate(row, from_attributes=True) for row in result_orm]
    print(f'\n\n{result_dto=}\n\n')

2026-01-29 14:39:52,226 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-29 14:39:52,226 INFO sqlalchemy.engine.Engine SELECT employees_orm.id, employees_orm.username 
FROM employees_orm 
 LIMIT %(param_1)s::INTEGER
2026-01-29 14:39:52,227 INFO sqlalchemy.engine.Engine [cached since 168s ago] {'param_1': 2}


result_orm=[<EmployeesOrm id=2, username=Michael>, <EmployeesOrm id=1, username=Kurl>]




result_dto=[EmployeesDTO(username='Michael', id=2), EmployeesDTO(username='Kurl', id=1)]


2026-01-29 14:39:52,230 INFO sqlalchemy.engine.Engine ROLLBACK


In [9]:
with sync_session() as session:
    query = (
        select(EmployeesOrm)
        .options(selectinload(EmployeesOrm.resumes))
        .limit(2)
    )

    res = session.execute(query)
    result_orm = res.scalars().all()
    print(f'\n\n{result_orm=}\n\n')
    result_dto = [EmployeesRelDTO.model_validate(row, from_attributes=True) for row in result_orm]
    print(f'{result_dto=}\n\n')

2026-01-29 14:42:10,408 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-29 14:42:10,410 INFO sqlalchemy.engine.Engine SELECT employees_orm.id, employees_orm.username 
FROM employees_orm 
 LIMIT %(param_1)s::INTEGER
2026-01-29 14:42:10,410 INFO sqlalchemy.engine.Engine [generated in 0.00070s] {'param_1': 2}
2026-01-29 14:42:10,415 INFO sqlalchemy.engine.Engine SELECT resumes_orm.employee_id AS resumes_orm_employee_id, resumes_orm.id AS resumes_orm_id, resumes_orm.title AS resumes_orm_title, resumes_orm.compensation AS resumes_orm_compensation, resumes_orm.workload AS resumes_orm_workload, resumes_orm.created_at AS resumes_orm_created_at, resumes_orm.updated_at AS resumes_orm_updated_at 
FROM resumes_orm 
WHERE resumes_orm.employee_id IN (%(primary_keys_1)s::INTEGER, %(primary_keys_2)s::INTEGER)
2026-01-29 14:42:10,416 INFO sqlalchemy.engine.Engine [generated in 0.00087s] {'primary_keys_1': 2, 'primary_keys_2': 1}


result_orm=[<EmployeesOrm id=2, username=Michael>, <EmployeesOrm 

In [13]:
from pydantic import BaseModel

class WorkloadAvgCompensationDTO(BaseModel):
    workload: 'Workload'
    avg_compensation: int

In [15]:
with sync_session() as session:
    query = (
        select(
            ResumesOrm.workload,
            cast(func.avg(ResumesOrm.compensation), Integer).label('avg_compensation'),
        )
        .select_from(ResumesOrm)
        .filter(and_(
            ResumesOrm.title.contains('Python'),
            ResumesOrm.compensation > 40000,
        ))
        .group_by(ResumesOrm.workload)
        .having(cast(func.avg(ResumesOrm.compensation), Integer) > 70000) 
    )
    print(f'\n\n{query}\n\n')
    print(f'\n\n{query.compile(compile_kwargs={"literal_binds": True})}\n\n')
    res = session.execute(query)
    result_orm = res.all()
    print(f'\n\n{result_orm=}\n\n')
    print(f'{result_orm[0].avg_compensation}\n\n')
    result_dto = [WorkloadAvgCompensationDTO.model_validate(row, from_attributes=True) for row in result_orm]
    print(f'{result_dto=}\n\n')



SELECT resumes_orm.workload, CAST(avg(resumes_orm.compensation) AS INTEGER) AS avg_compensation 
FROM resumes_orm 
WHERE (resumes_orm.title LIKE '%' || :title_1 || '%') AND resumes_orm.compensation > :compensation_1 GROUP BY resumes_orm.workload 
HAVING CAST(avg(resumes_orm.compensation) AS INTEGER) > :param_1




SELECT resumes_orm.workload, CAST(avg(resumes_orm.compensation) AS INTEGER) AS avg_compensation 
FROM resumes_orm 
WHERE (resumes_orm.title LIKE '%' || 'Python' || '%') AND resumes_orm.compensation > 40000 GROUP BY resumes_orm.workload 
HAVING CAST(avg(resumes_orm.compensation) AS INTEGER) > 70000


2026-01-29 14:52:31,123 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-29 14:52:31,124 INFO sqlalchemy.engine.Engine SELECT resumes_orm.workload, CAST(avg(resumes_orm.compensation) AS INTEGER) AS avg_compensation 
FROM resumes_orm 
WHERE (resumes_orm.title LIKE '%%' || %(title_1)s::VARCHAR || '%%') AND resumes_orm.compensation > %(compensation_1)s::INTEGER GROUP BY resum